## Setup

In [1]:
from typing import cast

from gitsource import GithubRepositoryDataReader

from lib.types import LessonDocument


reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = cast(list[LessonDocument], [file.parse() for file in reader.read()])

## Generate Ground Truth

In [2]:
from dotenv import dotenv_values
from openai import OpenAI


config = dotenv_values()

openai_client = OpenAI(api_key=config["OPENAI_API_KEY"])

In [3]:
how_to_generate_questions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [4]:
from pydantic import BaseModel


class Questions(BaseModel):
    questions: list[str]

## Q1. Generating questions

Generate questions for:
- 01-agentic-rag/lessons/01-intro.md
- 01-agentic-rag/lessons/02-environment.md
- 01-agentic-rag/lessons/03-rag.md

What's the average number of input tokens across these 3 calls?

- ❌ 140
- ✅ **1400**
- ❌ 14000
- ❌ 140000

> These numbers vary between runs, even with the same model, so pick the closest option. A different provider or model may land further apart, but the input tokens stay in the same order of magnitude - the prompt we send is the same.

In [5]:
print("filename:", documents[0]["filename"])

print("content:", documents[0]["content"])

filename: 01-agentic-rag/lessons/01-intro.md
content: # Introduction

Video: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In this module, we'll build a working Retrieval-Augmented
Generation (RAG) system from scratch, step by step.

We write everything in plain Python. We build a small search index by
hand and call the LLM ourselves. I want you to see every piece first.
That way you know what a framework does for you before you reach for
one.

Places where you can find me:

- [My substack](https://alexeyondata.substack.com/)
- [LinkedIn](https://www.linkedin.com/in/agrigorev/)
- [X](https://x.com/Al_Grigor)

## LLMs

An LLM (Large Language Model) is a neural network trained on massive
amounts of text. Given a prompt, it generates a continuation - a
plausible next piece of text.

Think of your phone. When you type "how are" in WhatsApp, it suggests
"you" as the next word. "How are you" is the most common continuation.
Your pho

In [6]:
import json

from lib.llm import call_structured_llm_with_retry


for document in documents[:3]:

    print(f"document: {document['filename']}")

    user_prompt = json.dumps(document)

    out, usage = call_structured_llm_with_retry(
        openai_client,
        instructions=how_to_generate_questions,
        user_prompt=user_prompt,
        output_type=Questions
    )

    print(out)
    print(usage)

document: 01-agentic-rag/lessons/01-intro.md
questions=['What exactly is a Retrieval-Augmented Generation system, and how does it help with wrong or missing LLM answers?', 'Why does this course treat the language model like a black box instead of explaining how it works inside?', 'What are the main limits of LLMs that make RAG useful in the first place?', 'What kind of project will this module build to demonstrate RAG in practice?', 'What will be covered in the first part of the module before the pipeline becomes agent-driven?']
ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=111, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1131)
document: 01-agentic-rag/lessons/02-environment.md
questions=['What do I need installed before starting this module, and do I need anything beyond Python and Jupyter?', 'How do I set up a new project for the course from scratch with uv?', 'Which packages does the lesson te

## The full ground truth

In [7]:
import pandas as pd

from lib.types import HomeworkGroundTruthRecord


homework_ground_truth_df = pd.read_csv("data/homework_ground_truth.csv")
homework_ground_truth = cast(
    list[HomeworkGroundTruthRecord],
    homework_ground_truth_df.to_dict(orient="records"),
)

display(homework_ground_truth_df)

,question,filename
0,What exactly is a retrieval-augmented generati...,01-agentic-rag/lessons/01-intro.md
1,Why does this course build the RAG project in ...,01-agentic-rag/lessons/01-intro.md
2,What are the main weaknesses of large language...,01-agentic-rag/lessons/01-intro.md
3,What will the course build in the first part o...,01-agentic-rag/lessons/01-intro.md
4,What kind of example app are you building here...,01-agentic-rag/lessons/01-intro.md
...,...,...
355,How should I break up a long article or transc...,07-project-example/lessons/07-chunking.md
356,"When I have several blog posts or wiki pages, ...",07-project-example/lessons/07-chunking.md
357,"For a single long PDF or video transcript, wha...",07-project-example/lessons/07-chunking.md
358,If I’m dealing with a book or other really lon...,07-project-example/lessons/07-chunking.md


## Searching the chunks

In [8]:
from gitsource import chunk_documents

from lib.types import LessonChunk


chunks = cast(
    list[LessonChunk],
    chunk_documents(
        cast(list[dict[str, str]], documents),
        size=2000,
        step=1000,
    ),
)

In [9]:
from scripts.embedder import Embedder
from lib.search import HybridSearchTool, MinsearchLexicalSearchTool, MinsearchSemanticSearchTool
from lib.search_types import LexicalSearchConfig, SemanticSearchConfig
from lib.types import LessonChunk


minsearch_lexical_search_tool = MinsearchLexicalSearchTool[LessonChunk].from_documents(
    documents=chunks,
    text_fields=["content"],
    keyword_fields=["filename"],
    config=LexicalSearchConfig(num_results=5),
)


encoder = Embedder("models/Xenova/all-MiniLM-L6-v2")

minsearch_semantic_search_tool = MinsearchSemanticSearchTool[LessonChunk].from_documents(
    documents=chunks,
    encoder=encoder,
    text_fields=["content"],
    keyword_fields=["filename"],
    config=SemanticSearchConfig(num_results=5),
)

  0%|          | 0/6 [00:00<?, ?it/s]

In [10]:
# For hybrid search, each retriever should surface more candidates than the
# final result count so RRF has something to fuse - the homework has
# text_search/vector_search fetch 10 each before combining into top 5.
# Reuses the already-built index/encoder, so nothing gets re-embedded.
hybrid_lexical_search_tool = MinsearchLexicalSearchTool[LessonChunk](
    index=minsearch_lexical_search_tool.index,
    config=LexicalSearchConfig(num_results=10),
)

hybrid_semantic_search_tool = MinsearchSemanticSearchTool[LessonChunk](
    index=minsearch_semantic_search_tool.index,
    encoder=encoder,
    config=SemanticSearchConfig(num_results=10),
)

## Q2. First result with text search

Take the first question from the ground truth:

```python
q = ground_truth[0]["question"]
```

After running `text_search` for it, what's the `filename` of the first result?
- ❌ 01-agentic-rag/lessons/01-intro.md
- ✅ 01-agentic-rag/lessons/03-rag.md
- ❌ 01-agentic-rag/lessons/13-function-calling.md
- ❌ 01-agentic-rag/lessons/10-rag-next-steps.md

In [11]:
question = homework_ground_truth[0]["question"]

print(question)

What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?


In [12]:
lexical_search_result = minsearch_lexical_search_tool.search(question)

print(lexical_search_result[0]['filename'])

print(lexical_search_result[0]['content'])

01-agentic-rag/lessons/03-rag.md
we drop it.

Build a prompt that includes both the question and the context:

```python
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""
```

Instead of sending the raw question to the LLM, we send this prompt:

```python
answer = llm(prompt)
print(answer)
```

After that, the answer is correct: "Yes, you can still join. If you want to
receive a certificate, you need to submit your project while
submissions are still open."

This is the answer we actually want to give to our students. What we
just did is nothing but RAG.

## Retrieval plus generation

RAG stands for Retrieval-Augmented Generation. Generation is the LLM
producing text, and retrieval is search. We retrieve relevant documents
from our knowled

## Q3. First result with vector search

After running `vector_search` for the same question, what's the `filename` of the first result?
- ✅ 01-agentic-rag/lessons/01-intro.md
- ❌ 01-agentic-rag/lessons/03-rag.md
- ❌ 04-evaluation/lessons/11-evaluation-intro.md
- ❌ 04-evaluation/lessons/12-rag-answers.md

This question was generated from `01-agentic-rag/lessons/01-intro.md`. Notice that one method finds the right page at the top and the other doesn't. That's exactly why we measure across the whole dataset instead of trusting one query.

In [13]:
semantic_search_result = minsearch_semantic_search_tool.search(question)

print(semantic_search_result[0]['filename'])

print(semantic_search_result[0]['content'])

01-agentic-rag/lessons/01-intro.md
# Introduction

Video: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In this module, we'll build a working Retrieval-Augmented
Generation (RAG) system from scratch, step by step.

We write everything in plain Python. We build a small search index by
hand and call the LLM ourselves. I want you to see every piece first.
That way you know what a framework does for you before you reach for
one.

Places where you can find me:

- [My substack](https://alexeyondata.substack.com/)
- [LinkedIn](https://www.linkedin.com/in/agrigorev/)
- [X](https://x.com/Al_Grigor)

## LLMs

An LLM (Large Language Model) is a neural network trained on massive
amounts of text. Given a prompt, it generates a continuation - a
plausible next piece of text.

Think of your phone. When you type "how are" in WhatsApp, it suggests
"you" as the next word. "How are you" is the most common continuation.
Your phone uses a simple la

## Q4. Evaluating text search

Evaluate `text_search` on the ground truth data.

What's the Hit Rate?

- ❌ 0.55
- ❌ 0.66
- ✅ 0.76
- ❌ 0.88

In [14]:
from lib.search import SearchTool


def are_results_relevant(
        results: list[LessonChunk],
        ground_truth: HomeworkGroundTruthRecord) -> list[int]:
    return [int(result["filename"] == ground_truth["filename"]) for result in results]


def compute_total_relevance(
        homework_ground_truth: list[HomeworkGroundTruthRecord],
        search_tool: SearchTool[LessonChunk]
        ) -> list[list[int]]:
    total_relevance_matrix: list[list[int]] = []
    for ground_truth_record in homework_ground_truth:
        results = search_tool.search(ground_truth_record["question"])
        total_relevance_matrix.append(
            are_results_relevant(results, ground_truth_record))
    return total_relevance_matrix

In [15]:
def compute_hit_rate(relevances: list[list[int]]) -> float:
    relevance_per_question = [any(relevance) for relevance in relevances]
    hit_rate = sum(relevance_per_question) / len(relevances)
    return hit_rate

def compute_mrr(relevances: list[list[int]]) -> float:
    total_score = 0.0
    for relevance in relevances:
        for rank in range(len(relevance)):
            if relevance[rank] == 1:
                score = 1 / (rank + 1)
                total_score += score
                break
    return total_score / len(relevances)

In [16]:
def evaluate(
        homework_ground_truth: list[HomeworkGroundTruthRecord],
        search_function: SearchTool[LessonChunk]):
    
    total_relevance = compute_total_relevance(homework_ground_truth, search_function)

    return {
        "hit_rate": compute_hit_rate(total_relevance),
        "mrr": compute_mrr(total_relevance),
    }

In [17]:
evaluate(homework_ground_truth, minsearch_lexical_search_tool)

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

## Q5. Evaluating vector search

Now evaluate `vector_search` - the part we left for the homework, since the module only evaluated keyword search.
    
What's the MRR?

- ❌ 0.35
- ❌ 0.45
- ✅ 0.55
- ❌ 0.65


In [18]:
evaluate(homework_ground_truth, minsearch_semantic_search_tool)

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

## Q6. Tuning hybrid search

The `k` constant in RRF controls how much the top ranks matter. A smaller `k`
sharpens the gap between positions, so being at the top of a list counts for
more. The RRF paper uses 60 as a default, but the best value depends on the data - so let's measure it.

Evaluate `hybrid_search` over the full ground truth dataset for `k` values 1,
50, 100, and 200. Compare the MRR values for these runs.

Which `k` gives the best MRR?

* ✅ 1
* ❌ 50
* ❌ 100
* ❌ 200

> Several values of `k` may give the same MRR. If there's a tie, pick the
> smallest `k`.

In [19]:
ks_to_check = [1, 50, 100, 200]

for k in ks_to_check:

    hybrid_search_tool = HybridSearchTool[LessonChunk](
        hybrid_lexical_search_tool,
        hybrid_semantic_search_tool,
        key_fields=["filename", "start"],
        k=k,
        num_results=5,
    )

    print(f"k: {k}, {evaluate(homework_ground_truth, hybrid_search_tool)}")

k: 1, {'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449}
k: 50, {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}
k: 100, {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}
k: 200, {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}
